# Lab 11 — 몬테카를로와 Bootstrap

**확률통계 · Topic 11 · 부산대학교 정보컴퓨터공학부** | **배점 10점**

---

## ✍️ 제출자 정보 — 먼저 채우세요

| | |
|---|---|
| **학번** | (여기에 작성) |

> 파일명을 **`Lab_학번_T11.ipynb`** 로 바꿔서 제출한다. (예: `Lab_202512345_T11.ipynb`)
> ⚠️ **제출 방법과 기한은 PLATO · Google Classroom 공지**를 확인한다.

### 배점

| 할 일 | 내용 | 배점 |
|:-:|---|:-:|
| 실습 1 | 몬테카를로 적분과 표준오차 | 2 |
| 실습 2 | 오차의 로그-로그 기울기 확인 | 2 |
| 실습 3 | 고차원에서 중심 근처에 들 확률 | 2 |
| 실습 4 | Bootstrap 으로 중앙값의 95% 구간 | 2 |
| 실습 5 | 구간이 참값을 덮는지 판정 | 1 |
| 마무리 | 오늘 배운 것 한 문장 | 1 |
| | **합계** | **10** |

⚠️ **제출 전 `런타임 → 모두 실행`** 으로 출력을 남길 것. 출력이 없으면 채점할 수 없다.

---

### 오늘의 목표

1. 몬테카를로 오차가 $O(1/\sqrt{n})$ 임을 로그-로그 기울기로 확인한다.
2. **고차원**에서 격자 방식이 무너지고 몬테카를로가 살아남는 것을 본다.
3. **Bootstrap**으로 중앙값의 신뢰구간을 구하고, **언제 무너지는지** 관찰한다.

⏱ **예상 소요 시간: 35분**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260302)
print("준비 완료")

## Part 1. 몬테카를로로 적분하기

$$I = \int_0^1 e^{-x^2}\,dx$$

손으로는 못 푸는 적분이다. **기댓값 → 표본평균**으로 바꾼다.

### ✏️ 실습 1 — 몬테카를로 적분과 표준오차 **(2점)**

In [ ]:
def mc_integral(n, rng):
    u = rng.random(n)
    g = np.exp(-u ** 2)
    # TODO 1: 추정값(표본평균)과 표준오차(std/sqrt(n))를 돌려주세요
    est = 0.0
    se = 0.0
    return est, se


for n in [1000, 100_000, 1_000_000]:
    est, se = mc_integral(n, rng)
    print(f"n={n:>9,}  추정 {est:.5f} +- {1.96 * se:.5f}")

print("\n참값 (scipy 수치적분): 0.746824")

> **추정값만 보지 말고 구간을 보자.** 참값이 구간 안에 들어오는가?
> 몬테카를로 결과는 **항상 오차와 함께** 보고해야 한다.

### ✏️ 실습 2 — 오차의 로그-로그 기울기 확인 **(2점)**

In [ ]:
TRUE = 0.7468241328124271
ns = np.unique(np.logspace(2, 6, 25).astype(int))
errs = [abs(mc_integral(n, rng)[0] - TRUE) for n in ns]

plt.figure(figsize=(6.5, 4.2))
plt.loglog(ns, errs, "o", ms=6, label="actual error")
plt.loglog(ns, 0.3 / np.sqrt(ns), lw=2, color="red", label="C/sqrt(n)")
plt.xlabel("n")
plt.ylabel("|estimate - true|")
plt.title("Monte Carlo error")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.show()

# TODO 2: 로그-로그 기울기를 구하세요
#         힌트: np.polyfit(np.log(ns), np.log(np.maximum(errs, 1e-12)), 1)[0]
slope = 0.0
print(f"로그-로그 기울기: {slope:.3f}   (이론 -0.5)")

## Part 2. 고차원에서의 승부

$X_1, \dots, X_d$ 를 $U(-\tfrac12, \tfrac12)$ 에서 독립으로 뽑을 때, 점이 중심에서 ½ 안에 들 확률
$\mathbb{P}[X_1^2 + \cdots + X_d^2 \le \tfrac14]$ 를 구한다. ($d=2$ 면 정사각형 속 원, $d=3$ 이면 정육면체 속 구의 넓이·부피 비율이다.)

- **격자 방식**: 축마다 $k$ 개 점 → 전체 $k^d$ 개. $d$ 가 커지면 폭발한다
- **몬테카를로**: 그냥 $n$ 개 뽑는다. $d$ 는 상관없다

### ✏️ 실습 3 — 고차원에서 중심 근처에 들 확률 **(2점)**

In [ ]:
def mc_ball_fraction(d, n, rng):
    pts = rng.uniform(-0.5, 0.5, size=(n, d))
    # TODO 3: 원점에서의 거리 제곱이 1/4 이하인 점의 비율을 구하세요
    #         힌트: ((pts ** 2).sum(axis=1) <= 0.25).mean()
    return 0.0


print(f"{'d':>4}{'격자 필요 점수 (축당 100)':>28}{'MC 추정 (n=200,000)':>24}")
for d in [1, 2, 3, 5, 10, 20]:
    grid = 100 ** d
    frac = mc_ball_fraction(d, 200_000, rng)
    grid_str = f"10^{d * 2}" if d > 3 else f"{grid:,}"
    print(f"{d:>4}{grid_str:>28}{frac:>24.5f}")

🤯 두 가지가 동시에 보인다. 하나는 MC의 강점이고, 하나는 MC의 함정이다.

1. **격자 방식은 $d=10$ 만 되어도 $10^{20}$ 개** — 계산 불가능. 몬테카를로는 20만 개로 끝난다 (슬라이드 16)
2. **그런데 $d=20$ 의 MC 추정은 `0.00000` 이다.** 확률이 0이라서가 아니다 (참값 $2.46 \times 10^{-8}$).
   20만 개를 뿌려도 기대 적중 수가 $200{,}000 \times 2.46 \times 10^{-8} \approx 0.005$ 개라 **하나도 안 들어간 것**이다.
   고차원에서는 무작위 점이 거의 다 중심에서 먼 "구석"에 떨어진다.

두 번째가 바로 **rejection sampling의 채택률 붕괴**다 (슬라이드 22). 상자에 뿌린 점 중 목표 영역에 드는 비율이
채택률이므로, $d=20$ 이면 표본 하나를 얻는 데 평균 4천만 번을 던져야 한다.
MC 오차가 줄어드는 **속도** $1/\sqrt{n}$ 에는 $d$ 가 없지만, 재려는 확률이 아주 작으면 그 확률에 비해
오차가 엄청나게 크다. 그래서 고차원에서는 "어디에 점을 뿌릴지"를 바꾸는 방법(MCMC 등)이 필요해진다.

## Part 3. Bootstrap

이제 도구를 바꾼다. **표본 하나로 불확실성을 재는** 방법이다.

### ✏️ 실습 4 — Bootstrap 으로 중앙값의 95% 구간 **(2점)**

In [ ]:
data = rng.lognormal(mean=1.0, sigma=0.8, size=60)
print(f"표본 크기 {len(data)}, 관측 중앙값 {np.median(data):.4f}")

B = 5000
# TODO 4: 복원추출로 재표본을 만들고 중앙값을 B번 계산하세요
#         힌트: np.median(rng.choice(data, size=len(data), replace=True))
boot = np.array([np.median(data) for _ in range(B)])

lo, hi = np.percentile(boot, [2.5, 97.5])

plt.figure(figsize=(7, 4))
plt.hist(boot, bins=50, alpha=0.85)
plt.axvline(np.median(data), color="black", lw=2, label="observed median")
plt.axvline(lo, color="red", ls="--", lw=2, label="95% CI")
plt.axvline(hi, color="red", ls="--", lw=2)
plt.xlabel("bootstrap median")
plt.ylabel("count")
plt.title("Bootstrap distribution of the median")
plt.legend()
plt.show()

print(f"bootstrap 표준오차 {boot.std():.4f}")
print(f"95% 신뢰구간 [{lo:.4f}, {hi:.4f}]")

> ⚠️ **`replace=True` 가 핵심이다.** 이걸 빠뜨리면 매번 같은 표본이 나와
> 표준오차가 0이 되어버린다. (위 배포용 코드의 초기값이 정확히 그 상태다)

### ✏️ 실습 5 — 구간이 참값을 덮는지 판정 **(1점)**

Bootstrap은 언제 무너지는가?

표본 크기를 줄여가며 신뢰구간이 어떻게 되는지 본다.
**참값**을 알고 있으므로(우리가 만든 데이터니까) 구간이 참값을 덮는지 확인할 수 있다.

In [ ]:
TRUE_MEDIAN = np.exp(1.0)
print(f"참 중앙값 = {TRUE_MEDIAN:.4f}\n")

for n in [10, 30, 100, 500]:
    sample = rng.lognormal(mean=1.0, sigma=0.8, size=n)
    boot = np.array([np.median(rng.choice(sample, size=n, replace=True))
                     for _ in range(2000)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    # TODO 5: 참값이 구간 안에 들어오는지 판정하세요.  힌트: lo <= TRUE_MEDIAN <= hi
    covered = False
    print(f"n={n:>4}  구간 [{lo:6.3f}, {hi:6.3f}]  폭 {hi - lo:5.3f}  "
          f"참값 포함: {'O' if covered else 'X'}")

🤔 **$n$ 이 커질수록 구간이 좁아진다.** 폭이 대략 $1/\sqrt{n}$ 로 줄어드는지 확인해보자.

($n$ 이 50배 커지면 폭은 약 $\sqrt{50} \approx 7$배 좁아져야 한다)

$n=10$ 의 구간을 보자. 참값을 덮기는 했지만 **폭이 3을 넘는다** —
"중앙값은 1.5와 4.9 사이 어딘가"라는 말은 사실상 아무 정보도 주지 않는다.

**`size=10` 부분의 시드를 바꿔가며 여러 번 돌려보자.** $n=10$ 에서는 참값을
**놓치는 경우도 나온다.** 원래 표본이 우연히 치우치면 bootstrap은 그 치우침을 그대로 물려받는다.

> **Bootstrap은 데이터에 없는 정보를 만들어내지 못한다.**
> 다만 "내가 가진 데이터로 이만큼밖에 말할 수 없다"를 **정직하게** 알려준다.

---

## 마무리 — 오늘 배운 것 **(1점)**

**재표집 횟수 $B$ 를 늘리는 것과 표본 크기 $n$ 을 늘리는 것은 무엇이 다른가? 한 문장으로 써보자.**
이 셀을 더블클릭해서 바로 쓰면 된다.

> (여기에 작성)

---

## ✅ 제출 전 점검

- [ ] 맨 위에 **학번**을 적었다
- [ ] 실습 1~5 의 `# TODO` 를 모두 채웠다
- [ ] `런타임 → 모두 실행` 으로 **모든 출력이 남아 있다**
- [ ] 그래프에 축 이름 · 제목이 있고 **한글이 없다**
- [ ] 위 한 문장을 작성했다
- [ ] 파일명을 **`Lab_학번_T11.ipynb`** 로 바꿨다

**제출처와 기한은 PLATO · Google Classroom 공지를 확인한다.**

### 스스로 확인 (채점 대상 아님)

- 적분을 기댓값으로 바꿔 몬테카를로로 풀 수 있다
- 몬테카를로 결과에 95% 구간을 붙일 수 있다
- 고차원에서 격자 방식이 왜 무너지는지, 그리고 $d=20$ 의 `0.00000` 이 왜 나오는지 설명할 수 있다
- Bootstrap으로 공식이 없는 통계량의 신뢰구간을 구할 수 있다
- `replace=True` 를 빠뜨리면 안 되는 이유를 안다

> 📌 **미니 프로젝트 2** (`hw/mini_project_2.md`) — 오늘 배운 bootstrap이 핵심 도구다.